In [ ]:
# Auto-injected: export every displayed figure as PNG + PDF + SVG into png/ pdf/ svg/
import os
import matplotlib.pyplot as plt
_PDF_STEM = 'fig3_4'
_PDF_N = [0]
for _d in ('pdf', 'svg', 'png'):
    os.makedirs(_d, exist_ok=True)
def _save_fig(base, fig=None, dpi=300, **kw):
    f = fig if fig is not None else plt.gcf()
    f.savefig(f'pdf/{base}.pdf', bbox_inches='tight', **kw)
    f.savefig(f'svg/{base}.svg', bbox_inches='tight', **kw)
    f.savefig(f'png/{base}.png', dpi=dpi, bbox_inches='tight', **kw)
def _save_pdf():
    _PDF_N[0] += 1
    _save_fig(f'{_PDF_STEM}_{_PDF_N[0]:02d}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from pathlib import Path
import geopandas as gpd
from shapely.geometry import Point
import seaborn as sns # 引入 seaborn 来使用更专业的调色板
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy import stats

# 设置字体为Arial
plt.rcParams['font.family'] = 'Arial'

# 读取可行性分析结果数据
data_path = Path("../result/island_viability_summary_electric.csv")
df = pd.read_csv(data_path)

# 显示数据概况
print(f"数据形状: {df.shape}")
print(f"情景列表: {df['scenario'].unique()}")
print(f"Viability gap统计:")
print(df['viability_gap'].describe())



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Polygon, Rectangle # 导入多边形和矩形

# --- 全局字体和绘图设置 ---
plt.rcParams['font.family'] = 'Arial'  # 设置全局字体为Arial
plt.rcParams['font.size'] = 28  # 设置全局默认字体大小
plt.rcParams['axes.labelsize'] = 28  # 设置轴标签字体大小
plt.rcParams['xtick.labelsize'] = 28  # 设置x轴刻度标签字体大小
plt.rcParams['ytick.labelsize'] = 20  # 设置y轴刻度标签字体大小
plt.rcParams['legend.fontsize'] = 28  # 设置图例字体大小
plt.rcParams['figure.dpi'] = 300  # 设置图像分辨率

# --- 六区域定义 (保持不变) ---
SIX_REGION_NAMES = [
    "Feasible\nLow Cost\nHigh Affordability",      # 区域1
    "Feasible\nHigh Cost\nHigh Affordability",     # 区域2
    "Feasible\nLow Cost\nLow Affordability",       # 区域3
    "Infeasible\nHigh Cost\nHigh Affordability",   # 区域4
    "Infeasible\nLow Cost\nLow Affordability",     # 区域5
    "Infeasible\nHigh Cost\nLow Affordability",    # 区域6
]

# --- 核心分类函数 (保持不变) ---
def classify_point(x, y, median_breakeven):
    """根据成本中位数对单个点(x,y)进行六区域分类"""
    is_low_cost = x <= median_breakeven
    is_high_affordability_benchmark = y > median_breakeven
    is_feasible = y >= x
    if is_feasible:
        if is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[0]
        elif not is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[1]
        elif is_low_cost and not is_high_affordability_benchmark: return SIX_REGION_NAMES[2]
        else: return SIX_REGION_NAMES[5]
    else: # Infeasible
        if not is_low_cost and is_high_affordability_benchmark: return SIX_REGION_NAMES[3]
        elif is_low_cost and not is_high_affordability_benchmark: return SIX_REGION_NAMES[4]
        elif not is_low_cost and not is_high_affordability_benchmark: return SIX_REGION_NAMES[5]
        else: return SIX_REGION_NAMES[3]

# <<<< NEW: 定义Q1-Q6的颜色 >>>>
# 根据您的六区域颜色进行映射
Q_COLORS = {
    "Q1": '#012A61', # 深蓝 (最理想)
    "Q2": '#0B75B3', # 中蓝
    "Q3": '#89CAEA', # 浅蓝
    "Q4": "#F0D2D2", # 浅红
    "Q5": '#DC5654', # 中红
    "Q6": '#982B2D', # 深红
}


# <<<< MODIFIED: 主函数现在绘制连接式堆叠柱状图 >>>>
def analyze_and_plot_statistics(df_analysis, scenario_config, median_breakeven, label_mapping):
    """
    主函数：进行六区域分类，绘制连接式堆叠柱状图。
    """
    results = []
    ordered_q_labels = sorted(list(set(label_mapping.values())))

    # 1. 数据统计 (保持不变)
    for scenario_name, config in scenario_config.items():
        scenario_df = df_analysis[df_analysis['scenario'] == scenario_name].copy()
        if scenario_df.empty:
            continue
        scenario_df['region'] = scenario_df.apply(
            lambda row: classify_point(row['tariff_breakeven'], row['tariff_affordable'], median_breakeven),
            axis=1
        )
        counts = scenario_df['region'].value_counts().to_dict()
        q_counts = {q_label: 0 for q_label in ordered_q_labels}
        for region_name, q_label in label_mapping.items():
            q_counts[q_label] += counts.get(region_name, 0)
        result_row = {'Scenario': config['label'], **q_counts}
        results.append(result_row)
        
    if not results:
        print("没有可供分析的数据。")
        return
        
    # 2. 创建统计DataFrame (保持不变)
    stats_df = pd.DataFrame(results).set_index('Scenario')
    # 确保列的顺序是 Q1, Q2, ...
    stats_df = stats_df[ordered_q_labels]
    print("\n--- Q1-Q6 统计分析结果 ---")
    print(stats_df)
    print("-" * 50)
    
    # 3. 绘制连接式堆叠柱状图
    fig, ax = plt.subplots(figsize=(12, 7))  # 创建图形和轴对象
    scenarios = stats_df.index.tolist()
    categories = stats_df.columns.tolist()
    
    # --- 柱状图位置和尺寸设置 ---
    bar_width = 0.6  # 固定柱子宽度
    
    # 自定义x坐标位置 - 让Ideal和Climate Stress离两端更远
    x_positions = [1.2, 3.0, 4.8]  # 三个柱子的x坐标位置
    # 这样设置让柱子都远离x轴两端，中间间距保持均匀
    
    # 用于存储每个色块的Y轴坐标 (bottom, top)
    segment_coords = {scenario: {} for scenario in scenarios}

    # 绘制堆叠柱状图
    for i, scenario in enumerate(scenarios):
        bottom = 0
        x_pos = x_positions[i]  # 使用自定义的x坐标位置
        for category in categories:
            count = stats_df.loc[scenario, category]
            ax.bar(x_pos, count, bottom=bottom, width=bar_width,
                   label=category, color=Q_COLORS.get(category, 'gray'))
            segment_coords[scenario][category] = (bottom, bottom + count)
            bottom += count

    # 绘制连接多边形
    for i in range(len(scenarios) - 1):
        scen1 = scenarios[i]
        scen2 = scenarios[i+1]
        
        x1 = x_positions[i]      # 当前柱子的x坐标
        x2 = x_positions[i+1]    # 下一个柱子的x坐标
        
        for category in categories:
            y_bottom1, y_top1 = segment_coords[scen1][category]
            y_bottom2, y_top2 = segment_coords[scen2][category]
            
            # 定义多边形的四个顶点
            verts = [
                (x1 + bar_width/2, y_bottom1),    # 左下角
                (x1 + bar_width/2, y_top1),       # 左上角
                (x2 - bar_width/2, y_top2),       # 右上角
                (x2 - bar_width/2, y_bottom2)     # 右下角
            ]
            
            poly = Polygon(verts, facecolor=Q_COLORS.get(category, 'gray'), alpha=0.4, edgecolor=None)
            ax.add_patch(poly)

    # 4. 美化图表
    ax.set_ylabel('Number of Islands')  # Y轴标签
    # ax.set_xlabel('Scenario')  # X轴标签
    ax.set_xticks(x_positions)  # 使用自定义的x坐标位置
    ax.set_xticklabels(scenarios)  # X轴刻度标签
    ax.spines[['right', 'top']].set_visible(False)  # 隐藏右侧和顶部边框
    
    # 设置x轴范围，让柱子离两端更远
    ax.set_xlim(0, 6)  # 固定x轴范围，让柱子远离两端
    ax.set_ylim(-50, 1900)
    # 5. 创建图例 (图例现在代表Q1-Q6)
    # legend_elements = [Rectangle((0, 0), 1, 1, facecolor=Q_COLORS.get(cat, 'gray'), label=cat)
    #                    for cat in categories]
    # legend = ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.15),
    #                  ncol=len(categories), frameon=False)

    fig.tight_layout(rect=[0, 0, 1, 0.96])  # 调整布局为图例留出空间
    _save_pdf()
    plt.show()

# --- 主程序调用 ---
if __name__ == '__main__':
    df_base = df[df['scenario'] == 'output_0']
    median_breakeven = df_base['tariff_breakeven'].median()
    
    # 模拟情景配置
    scenario_config = {
        'output_0': {'label': 'Ideal', 'color': '#1f77b4'},
        'output_2020': {'label': 'Baseline', 'color': '#ff7f0e'},
        'output_2050': {'label': 'Climate Stress', 'color': '#009988'}
    }
    
    # 模拟六区域到Q1-Q6的映射
    label_mapping = {
        SIX_REGION_NAMES[0]: "Q1", SIX_REGION_NAMES[1]: "Q2",
        SIX_REGION_NAMES[2]: "Q3", SIX_REGION_NAMES[3]: "Q4",
        SIX_REGION_NAMES[4]: "Q5", SIX_REGION_NAMES[5]: "Q6"
    }
    
    # 调用修改后的函数
    analyze_and_plot_statistics(df, scenario_config, median_breakeven, label_mapping)

In [ ]:
# <<<< NEW: 单独输出图例的函数 >>>>
def create_separate_legend():
    """
    创建单独的图例文件
    """
    # Q1-Q6的颜色配置 (与主图保持一致)
    Q_COLORS = {
        "Q1": '#012A61', # 深蓝 (最理想)
        "Q2": '#0B75B3', # 中蓝
        "Q3": '#89CAEA', # 浅蓝
        "Q4": "#F0D2D2", # 浅红
        "Q5": '#DC5654', # 中红
        "Q6": '#982B2D', # 深红
    }

    categories = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"]  # 图例标签

    # 创建只包含图例的图形
    fig, ax = plt.subplots(figsize=(4, 2))  # 宽度较大，高度较小的图形尺寸

    # 隐藏坐标轴
    ax.set_xlim(0, 1)  # 设置x轴范围
    ax.set_ylim(0, 1)  # 设置y轴范围
    ax.axis('off')     # 隐藏所有坐标轴元素

    # 创建图例元素
    legend_elements = [Rectangle((0, 0), 1, 1, facecolor=Q_COLORS.get(cat, 'gray'), label=cat)
                       for cat in categories]

    # 创建水平排列的图例
    legend = ax.legend(handles=legend_elements,
                      loc='center',                    # 图例位置居中
                      bbox_to_anchor=(0.5, 0.5),      # 图例在图形中心
                      ncol=6,            # 水平排列所有图例项
                      frameon=False,                   # 不显示图例边框
                      handlelength=2,                  # 图例标记长度
                      handletextpad=0.5,               # 图例标记和文字间距
                      columnspacing=1.5)               # 图例列间距

    # 保存图例为独立文件
    # plt.savefig('fig4.4_legend.png',           # 文件名
    #             dpi=300,                       # 高分辨率输出
    #             bbox_inches='tight',           # 紧密裁剪边界
    #             facecolor='white',             # 背景色为白色
    #             edgecolor='none',              # 无边框颜色
    #             transparent=False)             # 不透明背景

    _save_pdf()
    plt.show()

# 调用图例输出函数
create_separate_legend()

In [ ]:
# <<<< NEW: 四柱图可视化 - Climate Stress + 三个Future情景 >>>>
def create_four_bar_chart():
    """
    创建包含Climate Stress和三个Future情景的四柱图
    """
    # 四个情景的配置
    four_scenario_config = {
        'output_2050': {'label': 'Climate \nStress', 'color': '#009988'},
        'output_future_2030': {'label': 'TP 2030', 'color': '#d62728'},
        'output_future_2040': {'label': 'TP 2040', 'color': '#ff7f0e'},
        'output_future_2050': {'label': 'TP 2050', 'color': '#1f77b4'}
    }
    
    # Q1-Q6的颜色配置 (与主图保持一致)
    Q_COLORS = {
        "Q1": '#012A61', # 深蓝 (最理想)
        "Q2": '#0B75B3', # 中蓝
        "Q3": '#89CAEA', # 浅蓝
        "Q4": "#F0D2D2", # 浅红
        "Q5": '#DC5654', # 中红
        "Q6": '#982B2D', # 深红
    }
    
    # 六区域到Q标签的映射
    label_mapping = {
        SIX_REGION_NAMES[0]: "Q1", SIX_REGION_NAMES[1]: "Q2",
        SIX_REGION_NAMES[2]: "Q3", SIX_REGION_NAMES[3]: "Q4",
        SIX_REGION_NAMES[4]: "Q5", SIX_REGION_NAMES[5]: "Q6"
    }
    
    # 基准数据计算中位数
    df_base = df[df['scenario'] == 'output_0']
    median_breakeven = df_base['tariff_breakeven'].median()
    
    # 数据处理和统计
    results = []
    ordered_q_labels = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"]
    
    for scenario_name, config in four_scenario_config.items():
        scenario_df = df[df['scenario'] == scenario_name].copy()
        if scenario_df.empty:
            continue
        
        # 进行六区域分类
        scenario_df['region'] = scenario_df.apply(
            lambda row: classify_point(row['tariff_breakeven'], row['tariff_affordable'], median_breakeven),
            axis=1
        )
        
        # 统计各区域数量
        counts = scenario_df['region'].value_counts().to_dict()
        q_counts = {q_label: 0 for q_label in ordered_q_labels}
        for region_name, q_label in label_mapping.items():
            q_counts[q_label] += counts.get(region_name, 0)
        
        result_row = {'Scenario': config['label'], **q_counts}
        results.append(result_row)
    
    if not results:
        print("没有可供分析的数据。")
        return
    
    # 创建统计DataFrame
    stats_df = pd.DataFrame(results).set_index('Scenario')
    stats_df = stats_df[ordered_q_labels]  # 确保列的顺序
    
    print("\\n--- 四情景 Q1-Q6 统计分析结果 ---")
    print(stats_df)
    print("-" * 50)
    
    # 绘制四柱图
    fig, ax = plt.subplots(figsize=(11, 6))  # 更宽的图形以容纳四个柱子
    
    scenarios = stats_df.index.tolist()
    categories = stats_df.columns.tolist()
    
    # 柱状图位置设置
    bar_width = 0.8  # 柱子宽度
    x_positions = [1, 3, 5, 7]  # 四个柱子的x坐标位置，间距均匀
    
    # 用于存储每个色块的Y轴坐标 (bottom, top) - 用于绘制连接线
    segment_coords = {scenario: {} for scenario in scenarios}
    
    # 绘制堆叠柱状图
    for i, scenario in enumerate(scenarios):
        bottom = 0
        x_pos = x_positions[i]
        for category in categories:
            count = stats_df.loc[scenario, category]
            ax.bar(x_pos, count, bottom=bottom, width=bar_width,
                   color=Q_COLORS.get(category, 'gray'))  # 移除label避免重复图例
            # 存储每个色块的坐标信息，用于绘制连接线
            segment_coords[scenario][category] = (bottom, bottom + count)
            bottom += count
    
    # <<<< NEW: 绘制连接多边形 (连接线) >>>>
    for i in range(len(scenarios) - 1):
        scen1 = scenarios[i]      # 当前柱子
        scen2 = scenarios[i+1]    # 下一个柱子
        
        x1 = x_positions[i]       # 当前柱子的x坐标
        x2 = x_positions[i+1]     # 下一个柱子的x坐标
        
        for category in categories:
            y_bottom1, y_top1 = segment_coords[scen1][category]  # 当前柱子该颜色块的y坐标
            y_bottom2, y_top2 = segment_coords[scen2][category]  # 下一个柱子该颜色块的y坐标
            
            # 定义连接多边形的四个顶点
            verts = [
                (x1 + bar_width/2, y_bottom1),    # 左下角
                (x1 + bar_width/2, y_top1),       # 左上角
                (x2 - bar_width/2, y_top2),       # 右上角
                (x2 - bar_width/2, y_bottom2)     # 右下角
            ]
            
            # 创建半透明的连接多边形
            poly = Polygon(verts, 
                          facecolor=Q_COLORS.get(category, 'gray'),  # 与对应色块相同颜色
                          alpha=0.4,                                  # 半透明效果
                          edgecolor=None)                             # 无边框
            ax.add_patch(poly)  # 添加多边形到图形中
    
    # 图表美化
    ax.set_ylabel('Number of Islands')  # Y轴标签
    ax.set_xticks(x_positions)  # 设置x轴刻度位置
    ax.set_xticklabels(scenarios)  # X轴标签
    ax.spines[['right', 'top']].set_visible(False)  # 隐藏右侧和顶部边框
    ax.set_xlim(-0.5, 8.5)  # 设置x轴范围，让柱子远离边界
    
    # 不在主图上显示图例
    fig.tight_layout()
    _save_pdf()
    plt.show()
    
    # 保存主图
    # plt.savefig('fig4.4_four_scenarios.png',    # 文件名
    #             dpi=300,                        # 高分辨率
    #             bbox_inches='tight',            # 紧密裁剪
    #             facecolor='white',              # 白色背景
    #             edgecolor='none')               # 无边框
    
    # print("四情景柱状图已保存为: fig4.4_four_scenarios.png")

# <<<< NEW: 四柱图对应的单独图例 >>>>
def create_four_scenario_legend():
    """
    为四柱图创建单独的图例文件
    """
    # Q1-Q6的颜色配置 (与主图保持一致)
    Q_COLORS = {
        "Q1": '#012A61', # 深蓝 (最理想)
        "Q2": '#0B75B3', # 中蓝
        "Q3": '#89CAEA', # 浅蓝
        "Q4": "#F0D2D2", # 浅红
        "Q5": '#DC5654', # 中红
        "Q6": '#982B2D', # 深红
    }

    categories = ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6"]  # 图例标签

    # 创建只包含图例的图形
    fig, ax = plt.subplots(figsize=(4, 2))  # 宽度较大，高度较小

    # 隐藏坐标轴
    ax.set_xlim(0, 1)  # x轴范围
    ax.set_ylim(0, 1)  # y轴范围
    ax.axis('off')     # 隐藏坐标轴

    # 创建图例元素
    legend_elements = [Rectangle((0, 0), 1, 1, facecolor=Q_COLORS.get(cat, 'gray'), label=cat)
                       for cat in categories]

    # 创建水平排列的图例
    legend = ax.legend(handles=legend_elements,
                      loc='center',                    # 居中位置
                      bbox_to_anchor=(0.5, 0.5),      # 图形中心
                      ncol=6,            # 水平排列
                      frameon=False,                   # 无边框
                      handlelength=2,                  # 图例标记长度
                      handletextpad=0.5,               # 标记文字间距
                      columnspacing=1.5)               # 列间距

    # 保存图例
    # plt.savefig('fig4.4_four_scenarios_legend.png',  # 文件名
    #             dpi=300,                              # 高分辨率
    #             bbox_inches='tight',                  # 紧密裁剪
    #             facecolor='white',                    # 白色背景
    #             edgecolor='none',                     # 无边框
    #             transparent=False)                    # 不透明

    _save_pdf()
    plt.show()

# 调用函数创建四柱图和对应图例
create_four_bar_chart()
create_four_scenario_legend()